# Load packages

In [1]:
import scanpy as sc
import pandas as pd

# Load Data

In [ ]:
data_raw = pd.read_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered.csv')

# Add annotations

In [4]:
# Create a new column celltype_1 with the values from celltype.1
data_raw['celltype_1'] = data_raw['celltype.1']

# Reclassify "Other" cells based on Vimentin expression
mask_other = data_raw['celltype_1'] == 'Other'
data_raw.loc[mask_other & (data_raw['Vimentin'] >= 0.5), 'celltype_1'] = 'VIM+.Stroma'
data_raw.loc[mask_other & (data_raw['Vimentin'] < 0.5), 'celltype_1'] = 'Other.Stroma'

# Change 'Immune' to 'Other_Immune'
# Change 'FOXP3.CD4.Tregs' to 'Treg'
data_raw.loc[data_raw['celltype_1'] == 'Immune', 'celltype_1'] = 'Other.Immune'
data_raw.loc[data_raw['celltype_1'] == 'FOXP3.CD4.Tregs', 'celltype_1'] = 'Treg'
data_raw.loc[data_raw['celltype_1'] == 'Myofibroblasts', 'celltype_1'] = 'SMA+.Stroma'

# Drop the original celltype.1 column
data_raw = data_raw.drop('celltype.1', axis=1)

In [6]:
# Create a new column Tumor_state with the values from celltype_1
data_raw['Tumor_state'] = data_raw['celltype_1']

# Reclassify "Tumor" cells based on Vimentin expression
mask_other = data_raw['Tumor_state'] == 'Tumor'
data_raw.loc[mask_other & (data_raw['Vimentin'] >= 0.5), 'Tumor_state'] = 'VIM+'
data_raw.loc[mask_other & (data_raw['Vimentin'] < 0.5), 'Tumor_state'] = 'Tumor'


In [13]:
data_raw['GCLC_VIM'] = data_raw['celltype_1']

mask_other = data_raw['GCLC_VIM'] == 'Tumor'
data_raw.loc[mask_other & (data_raw['GCLC'] >= 0.5) & (data_raw['Vimentin'] >= 0.5), 'GCLC_VIM'] = 'Tumor.GCLC+VIM+'
data_raw.loc[mask_other & (data_raw['GCLC'] >= 0.5) & (data_raw['Vimentin'] < 0.5), 'GCLC_VIM'] = 'Tumor.GCLC+VIM-'
data_raw.loc[mask_other & (data_raw['GCLC'] < 0.5) & (data_raw['Vimentin'] >= 0.5), 'GCLC_VIM'] = 'Tumor.GCLC-VIM+'
data_raw.loc[mask_other & (data_raw['GCLC'] < 0.5) & (data_raw['Vimentin'] < 0.5), 'GCLC_VIM'] = 'Tumor.GCLC-VIM-'

In [ ]:
Tumor = ['Tumor']
Stroma = ['SMA+.Stroma', 'VIM+.Stroma', 'Other.Stroma']
Immune = ['CD4.T.cells', 'CD8.T.cells', 'Treg', 'CD68.Macrophages', 'CD206.Macrophages', 'Other.Immune']

def determin_cell_category(celltype):
    if celltype in Tumor:
        return 'Tumor'
    if celltype in Stroma:
        return 'Stroma'
    if celltype in Immune:
        return 'Immune'
    return 'other'

data_raw['CellCategory'] = data_raw['celltype_1'].apply(determin_cell_category)
data_raw['CellCategory'].value_counts()

In [17]:
data_raw.to_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered_2.csv', index=False)